In [1]:
import numpy as np
import jax
from jax import jit, numpy as jnp

from itertools import product

In [23]:
strats = {
    "age": ["young","adult","older"],
    "disease_state": ["S","I","R"],
    "location": ["south", "north"]
}

comp_traits = list(product(*strats.values()))

ncomp = len(comp_traits)
comp_vals = np.ones(ncomp)
idx = np.arange(ncomp)

class CMap:

    def __init__(self, strats, comp_traits):
        self.strats = strats
        self.comp_traits = comp_traits
        self._strat_lookup = {k:i for i,k in enumerate(strats)}
        self.ncomp = len(comp_traits)

    @classmethod 
    def from_strats(cls, strats):
        comp_traits = np.array(list(product(*strats.values())))
        return cls(strats, comp_traits)

    def query(self, strat, stratum):
        strat_idx = self._strat_lookup[strat]
        out_indices = []
        for i, trait_table in enumerate(self.comp_traits):
            trait_val = trait_table[strat_idx]
            if trait_val == stratum:
                out_indices.append(i)
        return np.array(out_indices)
    
    def __getitem__(self, indices):
        comp_traits = self.comp_traits[indices]
        return CMap(self.strats, comp_traits)

    def __repr__(self):
        return "CMap: " + repr(self.comp_traits)
    
    def get_cats(self, strat):
        return [(strat, stratum) for stratum in self.strats[strat]]


cmap = CMap.from_strats(strats)

cmap[cmap.query("disease_state","I")]

CMap: array([['young', 'I', 'south'],
       ['young', 'I', 'north'],
       ['adult', 'I', 'south'],
       ['adult', 'I', 'north'],
       ['older', 'I', 'south'],
       ['older', 'I', 'north']], dtype='<U5')

In [73]:
cmap

CMap: array([['young', 'S', 'south'],
       ['young', 'S', 'north'],
       ['young', 'I', 'south'],
       ['young', 'I', 'north'],
       ['young', 'R', 'south'],
       ['young', 'R', 'north'],
       ['adult', 'S', 'south'],
       ['adult', 'S', 'north'],
       ['adult', 'I', 'south'],
       ['adult', 'I', 'north'],
       ['adult', 'R', 'south'],
       ['adult', 'R', 'north'],
       ['older', 'S', 'south'],
       ['older', 'S', 'north'],
       ['older', 'I', 'south'],
       ['older', 'I', 'north'],
       ['older', 'R', 'south'],
       ['older', 'R', 'north']], dtype='<U5')

In [24]:
class Flow:
    def __init__(self, src_map, dest_map):
        self.src_map = src_map
        self.dest_map = dest_map

    def get_compartment_indices(self, cmap):
        src_idx = cmap.query(*self.src_map)
        dest_idx = cmap.query(*self.dest_map)
        return src_idx, dest_idx

In [71]:
a = set(["a","c"])
b = set(["a","b"])

a | b, a - b, a and b


({'a', 'b', 'c'}, {'c'}, {'a', 'b'})

In [ ]:
replacement_birth : stratify deaths by region

In [26]:
cmap.get_cats("age")

[('age', 'young'), ('age', 'adult'), ('age', 'older')]

In [28]:
trait_cats = cmap.get_cats("age")

cmap.query(*trait_cats[0])

array([0, 1, 2, 3, 4, 5])

In [ ]:
class MulAdjustment:
    def __init__(self, trait_cats, adj_arr):
        self.trait_cats = trait_cats
        self.adj_arr = adj_arr
        self._adj_arr = adj_arr[:,np.newaxis]

    def adjust(self, cmap, vals):
        t_indices = []
        for trait in self.trait_cats:
            tidx = cmap.query(*trait)
            t_indices.append(tidx)
        t_indices = np.array(t_indices)
        return vals.at[t_indices].mul(self._adj_arr)


    

In [72]:
cmap.get_cats("age")

[('age', 'young'), ('age', 'adult'), ('age', 'older')]

In [ ]:
a = Adjustment(cmap.get_cats("age"), jnp.array((0.5,1.0,2.0)))

In [53]:
a.adjust(cmap, jnp.ones(cmap.ncomp))

Array([0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 1. , 1. , 1. , 1. , 1. , 1. , 2. ,
       2. , 2. , 2. , 2. , 2. ], dtype=float32)

In [50]:
np.array([0,1])[:,np.newaxis].shape

(2, 1)

In [57]:
x = jnp.array(([1,2,3,4]),dtype=float)
x.at[np.array([[0,2],[1,3]])].mul(np.array((1.2,1.3)).reshape(2,1))

Array([1.2      , 2.6      , 3.6000001, 5.2      ], dtype=float32)

In [9]:
comp_traits

[('young', 'S', 'south'),
 ('young', 'S', 'north'),
 ('young', 'I', 'south'),
 ('young', 'I', 'north'),
 ('young', 'R', 'south'),
 ('young', 'R', 'north'),
 ('adult', 'S', 'south'),
 ('adult', 'S', 'north'),
 ('adult', 'I', 'south'),
 ('adult', 'I', 'north'),
 ('adult', 'R', 'south'),
 ('adult', 'R', 'north'),
 ('older', 'S', 'south'),
 ('older', 'S', 'north'),
 ('older', 'I', 'south'),
 ('older', 'I', 'north'),
 ('older', 'R', 'south'),
 ('older', 'R', 'north')]

In [13]:
age_adj = np.array([1.2,1.5,2.0,1.7])

age_adj[prop_mask['age']]

array([1.2, 1.5, 2. , 1.7, 1.2, 1.5, 2. , 1.7, 1.2, 1.5, 2. , 1.7])

In [ ]:
class FlowSpec:
    def __init__(self, source, dest):
        self.source = source
        self.dest = dest

class AdjustmentLayer:
    def __init__(self, ):

class AdjustmentTable:
    def __init__(self):
        self.layers = []
